# 02 多输入 + 多样本的训练

使用 `x1`、`x2` 两个输入和多个样本，把多输入预测接入训练模板：
补全 **参数、损失、梯度与更新**。

真实关系设为 `y = 2.0*x1 - 3.0*x2 + 0.5`，让模型自己学出权重。

In [1]:
import torch

torch.manual_seed(0)

# 4 个样本，每个样本 2 个特征（x1, x2）
X = torch.tensor([
    [1.0, 0.0],
    [2.0, 1.0],
    [0.0, 1.0],
    [3.0, 2.0],
])
# 用真实关系 y = 2.0*x1 - 3.0*x2 + 0.5 算出的标签
y = torch.tensor([2.5, 1.5, -2.5, 0.5])

print('X.shape =', X.shape)   # (4, 2)：4个样本，每个2个特征
print('y.shape =', y.shape)   # (4,)

X.shape = torch.Size([4, 2])
y.shape = torch.Size([4])


### 补全参数
两个输入各需要一个权重，所以 `w` 是长度为 2 的向量；`b` 是一个偏置。

In [2]:
# 补全：参数 w（长度2）和 b（标量），都需要梯度
w = torch.randn(2, requires_grad=True)
b = torch.randn(1, requires_grad=True)
print('初始 w =', w.data)
print('初始 b =', b.data)

初始 w = tensor([ 1.5410, -0.2934])
初始 b = tensor([-2.1788])


### 训练循环
批量预测用矩阵乘法 `X @ w`：一次性算出所有样本的预测。

In [3]:
lr = 0.1
for epoch in range(100):
    # 补全预测：X @ w 形状为 (4,)，再加偏置 b
    y_pred = X @ w + b

    # 补全损失：均方误差
    loss = ((y_pred - y) ** 2).mean()

    # 反向传播，自动算出 w.grad 和 b.grad
    loss.backward()

    # 补全：用梯度更新参数，并清零梯度
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

    if (epoch + 1) % 20 == 0:
        print(f'epoch {epoch+1:3d}, loss = {loss.item():.6f}')

epoch  20, loss = 0.601648
epoch  40, loss = 0.107232
epoch  60, loss = 0.019156
epoch  80, loss = 0.003425
epoch 100, loss = 0.000613


In [4]:
# 查看学到的参数
print('学到的 w =', w.data, ' （真实值应为 [2.0, -3.0]）')
print('学到的 b =', b.data, ' （真实值应为 [0.5]）')
print('最终预测 =', (X @ w + b).data)
print('真实 y   =', y)

学到的 w = tensor([ 1.9914, -2.9611])  （真实值应为 [2.0, -3.0]）
学到的 b = tensor([0.4671])  （真实值应为 [0.5]）
最终预测 = tensor([ 2.4585,  1.4887, -2.4940,  0.5190])
真实 y   = tensor([ 2.5000,  1.5000, -2.5000,  0.5000])


## 观察

1. **参数形状**：`w` 是长度为 2 的向量，`w[0]` 对应 `x1` 的权重、`w[1]` 对应 `x2` 的权重；`b` 是标量偏置。
2. **批量预测**：`X @ w` 一次算出全部 4 个样本的预测（`X` 形状 `(4,2)`，`w` 形状 `(2,)`，结果形状 `(4,)`），不用写循环。
3. **损失下降**：随着训练轮数增加，loss 从较大值逐渐降到接近 0，说明模型在学习。
4. **学到的参数**：最终 `w` 接近 `[2.0, -3.0]`、`b` 接近 `0.5`，和我们设定的真实关系一致。
5. **梯度清零**：每次更新后必须 `w.grad.zero_()` / `b.grad.zero_()`，否则 PyTorch 会把新梯度累加到上次的梯度上，导致更新方向错误。